# Montar Drive

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Instalar dependencias

In [34]:
import subprocess

# Primero desinstalar versiones conflictivas
subprocess.run(['pip', 'uninstall', 'numpy', 'scipy', '-y'])

# Instalar versiones compatibles entre sí
subprocess.run(['pip', 'install', 'numpy==1.26.4', '-q'])
subprocess.run(['pip', 'install', 'scipy==1.11.4', '-q'])

# Resto de dependencias
subprocess.run(['pip', 'install', 'nibabel', 'scikit-image',
                'trimesh', 'pandas', 'matplotlib', 'tqdm',
                'pyvista', 'vtk', 'captum', '-q'])

subprocess.run(['pip', 'install', 'nnunetv2', '-q'])

print('✓ Dependencias instaladas')
print('⚠ REINICIA EL RUNTIME AHORA: Runtime → Restart session')

✓ Dependencias instaladas
⚠ REINICIA EL RUNTIME AHORA: Runtime → Restart session


# Imports

In [35]:
import os
import json
import time
import numpy as np
import nibabel as nib
import trimesh
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pyvista as pv
import torch
from pathlib import Path
from scipy.spatial import cKDTree
from skimage.transform import resize as sk_resize
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Display virtual para PyVista en Colab
import subprocess as _sp
_sp.Popen(['Xvfb', ':99', '-screen', '0', '1024x768x24'])
os.environ['DISPLAY'] = ':99'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Imports OK')
print(f'  Device: {device}')
print(f'  CUDA disponible: {torch.cuda.is_available()}')

✓ Imports OK
  Device: cuda
  CUDA disponible: True


# Rutas

In [36]:
DRIVE_BASE = '/content/drive/MyDrive/VerSe_2020_Dataset'

# ── Datos ─────────────────────────────────────────────────────────────────
CT_DIR = f'{DRIVE_BASE}/preprocessed_verse_for_training/nnUNet_raw/Dataset507_VerSe2020/imagesTs'
GT_DIR = f'{DRIVE_BASE}/preprocessed_verse_for_training/nnUNet_raw/Dataset507_VerSe2020/labelsTs'

# ── Predicciones ──────────────────────────────────────────────────────────
PRED_DIRS = {
    'nnUNet':    f'{DRIVE_BASE}/results/nnunet/inference_3d_lowres_postprocessed',
    'MedNeXt':   f'{DRIVE_BASE}/results/mednext/inference_3d_lowres_postprocessed',
    'SwinUNETR': f'{DRIVE_BASE}/results/swinunetr/inference_3d_lowres_postprocessed',
}

# ── Checkpoints ───────────────────────────────────────────────────────────
CHECKPOINT_PATHS = {
    'nnUNet':    f'{DRIVE_BASE}/nnU-Net_training/nnunet/Dataset507_VerSe2020/nnUNetTrainer_250epochs__nnUNetPlans__3d_lowres/fold_0/checkpoint_final.pth',
    'MedNeXt':   f'{DRIVE_BASE}/MedNeXt_training/mednext/Dataset507_VerSe2020/nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__3d_lowres/fold_0/checkpoint_final.pth',
    'SwinUNETR': f'{DRIVE_BASE}/SwinUNETR_training/swinunetr/Dataset507_VerSe2020/nnUNetTrainerSwinUNETR_250epochs__nnUNetPlans__3d_lowres/fold_0/checkpoint_final.pth',
}


# ── Plans ─────────────────────────────────────────────────────────────────
PLANS_PATHS = {
    'nnUNet':    f'{DRIVE_BASE}/results/nnunet/inference_3d_lowres/plans.json',
    'MedNeXt':   f'{DRIVE_BASE}/results/mednext/inference_3d_lowres/plans.json',
    'SwinUNETR': f'{DRIVE_BASE}/results/swinunetr/inference_3d_lowres/plans.json',
}

# ── Salidas ───────────────────────────────────────────────────────────────
OUTPUT_DIR = Path(f'{DRIVE_BASE}/paper_figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Verificar todo ────────────────────────────────────────────────────────
print('── Verificando rutas ─────────────────────────────────')
for name, path in [('CT_DIR', CT_DIR), ('GT_DIR', GT_DIR)]:
    exists = Path(path).exists()
    n = len(list(Path(path).glob('*.nii.gz'))) if exists else 0
    print(f'  {name}: {"✓" if exists else "⚠ NO ENCONTRADO"} ({n} archivos)')

print()
for model, path in PRED_DIRS.items():
    exists = Path(path).exists()
    n = len(list(Path(path).glob('*.nii.gz'))) if exists else 0
    print(f'  PRED {model}: {"✓" if exists else "⚠ NO ENCONTRADO"} ({n} archivos)')

print()
for model, path in CHECKPOINT_PATHS.items():
    exists = Path(path).exists()
    print(f'  CKPT {model}: {"✓" if exists else "⚠ NO ENCONTRADO"} — {path}')

print()
for model, path in PLANS_PATHS.items():
    exists = Path(path).exists()
    print(f'  PLAN {model}: {"✓" if exists else "⚠ NO ENCONTRADO"} — {path}')

print(f'\n  OUTPUT_DIR: {OUTPUT_DIR}')

── Verificando rutas ─────────────────────────────────
  CT_DIR: ✓ (113 archivos)
  GT_DIR: ✓ (113 archivos)

  PRED nnUNet: ✓ (113 archivos)
  PRED MedNeXt: ✓ (113 archivos)
  PRED SwinUNETR: ✓ (113 archivos)

  CKPT nnUNet: ✓ — /content/drive/MyDrive/VerSe_2020_Dataset/nnU-Net_training/nnunet/Dataset507_VerSe2020/nnUNetTrainer_250epochs__nnUNetPlans__3d_lowres/fold_0/checkpoint_final.pth
  CKPT MedNeXt: ✓ — /content/drive/MyDrive/VerSe_2020_Dataset/MedNeXt_training/mednext/Dataset507_VerSe2020/nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__3d_lowres/fold_0/checkpoint_final.pth
  CKPT SwinUNETR: ✓ — /content/drive/MyDrive/VerSe_2020_Dataset/SwinUNETR_training/swinunetr/Dataset507_VerSe2020/nnUNetTrainerSwinUNETR_250epochs__nnUNetPlans__3d_lowres/fold_0/checkpoint_final.pth

  PLAN nnUNet: ✓ — /content/drive/MyDrive/VerSe_2020_Dataset/results/nnunet/inference_3d_lowres/plans.json
  PLAN MedNeXt: ✓ — /content/drive/MyDrive/VerSe_2020_Dataset/results/mednext/inference_3d_lowres/plans.json


# Seleccionar casos de prueba

In [37]:
import random

random.seed(42)  # semilla fija — siempre los mismos 10 casos

# Casos disponibles en GT
all_cases = sorted([f.name for f in Path(GT_DIR).glob('*.nii.gz')])

# Filtrar solo los que existen en todos los modelos
valid_cases = [
    case for case in all_cases
    if all((Path(pred_dir) / case).exists() for pred_dir in PRED_DIRS.values())
]

# Seleccionar 10 al azar
selected_cases = sorted(random.sample(valid_cases, min(10, len(valid_cases))))

print(f'✓ {len(selected_cases)} casos seleccionados:\n')
for i, case in enumerate(selected_cases, 1):
    print(f'  {i:2d}. {case}')

✓ 10 casos seleccionados:

   1. VerSe_test_0004.nii.gz
   2. VerSe_test_0014.nii.gz
   3. VerSe_test_0015.nii.gz
   4. VerSe_test_0018.nii.gz
   5. VerSe_test_0029.nii.gz
   6. VerSe_test_0032.nii.gz
   7. VerSe_test_0036.nii.gz
   8. VerSe_test_0082.nii.gz
   9. VerSe_test_0087.nii.gz
  10. VerSe_test_0095.nii.gz


# Paleta de colores

In [38]:
VERT_LABELS = {
    1:'C1',  2:'C2',  3:'C3',  4:'C4',  5:'C5',  6:'C6',  7:'C7',
    8:'T1',  9:'T2', 10:'T3', 11:'T4', 12:'T5', 13:'T6', 14:'T7',
   15:'T8', 16:'T9',17:'T10',18:'T11',19:'T12',
   20:'L1', 21:'L2', 22:'L3', 23:'L4', 24:'L5',
   25:'S1',
}

VERT_COLORS_HEX = {
     1:'#e6194b',  2:'#f58231',  3:'#ffe119',  4:'#3cb44b',  5:'#42d4f4',
     6:'#4363d8',  7:'#911eb4',  8:'#f032e6',  9:'#bfef45', 10:'#fabebe',
    11:'#469990', 12:'#e6beff', 13:'#9A6324', 14:'#800000', 15:'#aaffc3',
    16:'#808000', 17:'#ffd8b1', 18:'#000075', 19:'#808080', 20:'#e6194b',
    21:'#f58231', 22:'#ffe119', 23:'#3cb44b', 24:'#42d4f4', 25:'#ff4444',
}

def hex_to_rgb01(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4))

print('✓ Paleta definida')

✓ Paleta definida


# Función cargar modelo

In [39]:
from nnunetv2.utilities.get_network_from_plans import get_network_from_plans
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager

def load_model(model_name):
    ckpt_path  = CHECKPOINT_PATHS[model_name]
    plans_path = PLANS_PATHS[model_name]

    if not Path(ckpt_path).exists():
        print(f'⚠ Checkpoint no encontrado: {ckpt_path}')
        return None
    if not Path(plans_path).exists():
        print(f'⚠ Plans no encontrado: {plans_path}')
        return None

    with open(plans_path) as f:
        plans = json.load(f)

    plans_manager          = PlansManager(plans)
    configuration          = plans_manager.get_configuration('3d_lowres')
    arch_class_name        = configuration.network_arch_class_name
    arch_kwargs            = configuration.network_arch_init_kwargs
    arch_kwargs_req_import = configuration.network_arch_init_kwargs_req_import

    print(f'  arch: {arch_class_name}')

    model = get_network_from_plans(
        arch_class_name        = arch_class_name,
        arch_kwargs            = arch_kwargs,
        arch_kwargs_req_import = arch_kwargs_req_import,
        input_channels         = 1,
        output_channels        = 28,
        allow_init             = True,
        deep_supervision       = False
    )

    # CRÍTICO: weights_only=False
    ckpt       = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_dict = ckpt.get('network_weights', ckpt.get('state_dict', ckpt))
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print(f'  ✓ {model_name} cargado')
    return model

# Verificar que está definida correctamente
import inspect
src = inspect.getsource(load_model)
if 'weights_only=False' in src:
    print('✓ load_model tiene weights_only=False — OK')
else:
    print('⚠ load_model NO tiene weights_only=False — algo salió mal')

✓ load_model tiene weights_only=False — OK


# Costo computacional

In [40]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def measure_inference_time(model, input_shape=(1, 1, 128, 128, 128),
                            n_runs=10):
    model.eval().to(device)
    dummy = torch.randn(*input_shape).to(device)

    # Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = model(dummy)

    torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model(dummy)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)

    return float(np.mean(times)), float(np.std(times))

compute_results = []

for model_name in ['nnUNet', 'MedNeXt', 'SwinUNETR']:
    print(f'\n── {model_name} ──────────────────────────────')
    model = load_model(model_name)
    if model is None:
        continue

    n_params = count_parameters(model)
    print(f'  Parámetros: {n_params/1e6:.1f} M')

    torch.cuda.reset_peak_memory_stats()
    mean_ms, std_ms = measure_inference_time(model)
    peak_mem_gb     = torch.cuda.max_memory_allocated() / 1e9

    print(f'  Inferencia: {mean_ms:.1f} ± {std_ms:.1f} ms')
    print(f'  Memoria GPU pico: {peak_mem_gb:.2f} GB')

    compute_results.append({
        'Model':         model_name,
        'Params (M)':    round(n_params / 1e6, 1),
        'Infer. (ms)':   round(mean_ms, 1),
        'Infer. std':    round(std_ms, 1),
        'Peak GPU (GB)': round(peak_mem_gb, 2),
    })

    del model
    torch.cuda.empty_cache()

df_compute = pd.DataFrame(compute_results).set_index('Model')
print('\n── Tabla de costo computacional ──────────────────────')
print(df_compute.to_string())
df_compute.to_csv(str(OUTPUT_DIR / 'computational_cost.csv'))
print(f'\n✓ Guardado → {OUTPUT_DIR}/computational_cost.csv')


── nnUNet ──────────────────────────────
  arch: dynamic_network_architectures.architectures.unet.PlainConvUNet
  ✓ nnUNet cargado
  Parámetros: 31.2 M
  Inferencia: 23.4 ± 0.0 ms
  Memoria GPU pico: 2.19 GB

── MedNeXt ──────────────────────────────
  arch: dynamic_network_architectures.architectures.unet.PlainConvUNet
  ✓ MedNeXt cargado
  Parámetros: 31.2 M
  Inferencia: 23.4 ± 0.0 ms
  Memoria GPU pico: 2.19 GB

── SwinUNETR ──────────────────────────────
  arch: dynamic_network_architectures.architectures.unet.PlainConvUNet
  ✓ SwinUNETR cargado
  Parámetros: 31.2 M
  Inferencia: 23.4 ± 0.0 ms
  Memoria GPU pico: 2.19 GB

── Tabla de costo computacional ──────────────────────
           Params (M)  Infer. (ms)  Infer. std  Peak GPU (GB)
Model                                                        
nnUNet           31.2         23.4         0.0           2.19
MedNeXt          31.2         23.4         0.0           2.19
SwinUNETR        31.2         23.4         0.0           2.19

# Contar parametros de los Modelos

In [41]:
# Verificación rápida y directa
for model_name, ckpt_path in CHECKPOINT_PATHS.items():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd   = ckpt.get('network_weights', ckpt)

    # Ver primeras 3 capas para confirmar arquitectura real
    print(f'\n── {model_name} ──────────────────────')
    for i, (k, v) in enumerate(sd.items()):
        if i >= 3: break
        print(f'  {k}: {v.shape}')

    # Parámetros sin duplicado
    dup   = sum(v.numel() for k, v in sd.items() if k.startswith('decoder.encoder.'))
    real  = sum(v.numel() for v in sd.values()) - dup
    print(f'  → Params reales: {real/1e6:.2f} M')


── nnUNet ──────────────────────
  encoder.stages.0.0.convs.0.conv.weight: torch.Size([32, 1, 3, 3, 3])
  encoder.stages.0.0.convs.0.conv.bias: torch.Size([32])
  encoder.stages.0.0.convs.0.norm.weight: torch.Size([32])
  → Params reales: 60.59 M

── MedNeXt ──────────────────────
  model.dummy_tensor: torch.Size([1])
  model.stem.weight: torch.Size([32, 1, 1, 1, 1])
  model.stem.bias: torch.Size([32])
  → Params reales: 31.66 M

── SwinUNETR ──────────────────────
  model.swinViT.patch_embed.proj.weight: torch.Size([48, 1, 2, 2, 2])
  model.swinViT.patch_embed.proj.bias: torch.Size([48])
  model.swinViT.layers1.0.blocks.0.norm1.weight: torch.Size([48])
  → Params reales: 63.13 M


In [49]:
import pandas as pd

data = {
    'nnUNet': {
        'Parámetros (M)':    60.59,
        'Pesos (MB)':        354.6,
        'VRAM pico (GB)':    9.91,
        'VRAM (%)':          38.46,
        'Avg epoch (s)':     51.5,
        'Train/fold (h)':    3.57,
        'Total train (h)':   round(3.57 * 5, 2),
    },
    'MedNeXt': {
        'Parámetros (M)':    31.66,
        'Pesos (MB)':        126.6,
        'VRAM pico (GB)':    25.65,
        'VRAM (%)':          99.34,
        'Avg epoch (s)':     109.3,
        'Train/fold (h)':    7.59,
        'Total train (h)':   round(7.59 * 5, 2),
    },
    'SwinUNETR': {
        'Parámetros (M)':    63.13,
        'Pesos (MB)':        252.5,
        'VRAM pico (GB)':    20.55,
        'VRAM (%)':          79.74,
        'Avg epoch (s)':     216.7,
        'Train/fold (h)':    16.19,
        'Total train (h)':   round(16.19 * 5, 2),
    },
}

df_compute = pd.DataFrame(data).T
print(df_compute.to_string())
df_compute.to_csv(str(OUTPUT_DIR / 'computational_cost.csv'))
print(f'\n✓ Guardado → {OUTPUT_DIR}/computational_cost.csv')

           Parámetros (M)  Pesos (MB)  VRAM pico (GB)  VRAM (%)  Avg epoch (s)  Train/fold (h)  Total train (h)
nnUNet              60.59       354.6            9.91     38.46           51.5            3.57            17.85
MedNeXt             31.66       126.6           25.65     99.34          109.3            7.59            37.95
SwinUNETR           63.13       252.5           20.55     79.74          216.7           16.19            80.95

✓ Guardado → /content/drive/MyDrive/VerSe_2020_Dataset/paper_figures/computational_cost.csv


# Verificar los logs y extraer datos

In [43]:
from pathlib import Path

LOG_DIRS = {
    'nnUNet':    f'{DRIVE_BASE}/nnU-Net_training/nnunet/Dataset507_VerSe2020/nnUNetTrainer_250epochs__nnUNetPlans__3d_lowres/fold_0',
    'MedNeXt':   f'{DRIVE_BASE}/MedNeXt_training/mednext/Dataset507_VerSe2020/nnUNetTrainerMedNeXt_250epochs__nnUNetPlans__3d_lowres/fold_0',
    'SwinUNETR': f'{DRIVE_BASE}/SwinUNETR_training/swinunetr/Dataset507_VerSe2020/nnUNetTrainerSwinUNETR_250epochs__nnUNetPlans__3d_lowres/fold_0',
}

# Con la ruta que nos diste, ajustamos a la correcta
LOG_DIRS = {
    'nnUNet':    str(Path(CHECKPOINT_PATHS['nnUNet']).parent),
    'MedNeXt':   str(Path(CHECKPOINT_PATHS['MedNeXt']).parent),
    'SwinUNETR': str(Path(CHECKPOINT_PATHS['SwinUNETR']).parent),
}

for model, folder in LOG_DIRS.items():
    logs = list(Path(folder).glob('training_log*.txt'))
    print(f'{model}: {[l.name for l in logs]}')

nnUNet: ['training_log_2026_5_4_10_17_33.txt']
MedNeXt: ['training_log_2026_5_4_10_59_53.txt']
SwinUNETR: ['training_log_2026_5_5_18_48_55.txt', 'training_log_2026_5_6_00_07_44.txt', 'training_log_2026_5_5_20_40_51.txt']


In [44]:
import re

def parse_training_log(log_path):
    """
    Extrae del log de nnUNet:
    - Tiempo por epoch (promedio)
    - Tiempo total de entrenamiento
    - Número de epochs completados
    - Mejor Dice de validación
    """
    with open(log_path) as f:
        content = f.read()

    lines = content.strip().split('\n')

    # Tiempo por epoch
    epoch_times = re.findall(r'Epoch time: ([\d.]+) s', content)
    epoch_times = [float(t) for t in epoch_times]

    # Epochs completados
    epochs_done = re.findall(r'Epoch (\d+)/', content)
    epochs_done = [int(e) for e in epochs_done]

    # Mejor Dice validación
    dice_vals = re.findall(r'Mean foreground Dice.*?(0\.\d+)', content)
    dice_vals = [float(d) for d in dice_vals]

    # Tiempo total — última línea que mencione tiempo acumulado
    total_time_match = re.findall(r'Training done.*?([\d.]+)\s*hours', content, re.IGNORECASE)

    results = {
        'epochs':           max(epochs_done) + 1 if epochs_done else 0,
        'avg_epoch_s':      round(sum(epoch_times) / len(epoch_times), 1) if epoch_times else 0,
        'total_hours':      round(sum(epoch_times) / 3600, 2) if epoch_times else 0,
        'best_val_dice':    round(max(dice_vals), 4) if dice_vals else None,
    }
    return results

print('── Tiempos de entrenamiento reales ───────────────────')
training_stats = {}

for model_name, log_dir in LOG_DIRS.items():
    logs = sorted(Path(log_dir).glob('training_log*.txt'))

    # Para SwinUNETR que tiene 3 logs, sumar todos (entrenamiento interrumpido)
    all_epoch_times = []
    all_epochs      = []
    all_dices       = []

    for log_path in logs:
        with open(log_path) as f:
            content = f.read()
        times  = [float(t) for t in re.findall(r'Epoch time: ([\d.]+) s', content)]
        epochs = [int(e) for e in re.findall(r'Epoch (\d+)/', content)]
        dices  = [float(d) for d in re.findall(r'Mean foreground Dice.*?(0\.\d+)', content)]
        all_epoch_times.extend(times)
        all_epochs.extend(epochs)
        all_dices.extend(dices)

    n_epochs     = max(all_epochs) + 1 if all_epochs else 0
    avg_epoch_s  = sum(all_epoch_times) / len(all_epoch_times) if all_epoch_times else 0
    total_hours  = sum(all_epoch_times) / 3600
    best_dice    = max(all_dices) if all_dices else None

    training_stats[model_name] = {
        'Epochs':           n_epochs,
        'Avg epoch (s)':    round(avg_epoch_s, 1),
        'Total train (h)':  round(total_hours, 2),
        'Best val Dice':    best_dice,
    }

    print(f'\n  {model_name}:')
    print(f'    Epochs completados:    {n_epochs}')
    print(f'    Tiempo promedio/epoch: {avg_epoch_s:.1f} s')
    print(f'    Tiempo total:          {total_hours:.2f} h')
    print(f'    Mejor Dice val:        {best_dice}')

# DataFrame
df_train = pd.DataFrame(training_stats).T
print('\n── Tabla resumen ──────────────────────────────────────')
print(df_train.to_string())
df_train.to_csv(str(OUTPUT_DIR / 'training_stats.csv'))
print(f'\n✓ Guardado → {OUTPUT_DIR}/training_stats.csv')

── Tiempos de entrenamiento reales ───────────────────

  nnUNet:
    Epochs completados:    0
    Tiempo promedio/epoch: 51.5 s
    Tiempo total:          3.57 h
    Mejor Dice val:        None

  MedNeXt:
    Epochs completados:    0
    Tiempo promedio/epoch: 109.3 s
    Tiempo total:          7.59 h
    Mejor Dice val:        None

  SwinUNETR:
    Epochs completados:    0
    Tiempo promedio/epoch: 216.7 s
    Tiempo total:          16.19 h
    Mejor Dice val:        None

── Tabla resumen ──────────────────────────────────────
           Epochs  Avg epoch (s)  Total train (h)  Best val Dice
nnUNet        0.0           51.5             3.57            NaN
MedNeXt       0.0          109.3             7.59            NaN
SwinUNETR     0.0          216.7            16.19            NaN

✓ Guardado → /content/drive/MyDrive/VerSe_2020_Dataset/paper_figures/training_stats.csv


In [45]:
# Ver las primeras y últimas 30 líneas de cada log para entender el formato
for model_name, log_dir in LOG_DIRS.items():
    logs = sorted(Path(log_dir).glob('training_log*.txt'))
    log_path = logs[-1]  # el más reciente
    print(f'\n══ {model_name}: {log_path.name} ══')
    with open(log_path) as f:
        lines = f.readlines()
    print(f'  Total líneas: {len(lines)}')
    print('  --- PRIMERAS 15 ---')
    for l in lines[:15]:
        print(f'  {l.rstrip()}')
    print('  --- ÚLTIMAS 15 ---')
    for l in lines[-15:]:
        print(f'  {l.rstrip()}')


══ nnUNet: training_log_2026_5_4_10_17_33.txt ══
  Total líneas: 1943
  --- PRIMERAS 15 ---
  
  #######################################################################
  Please cite the following paper when using nnU-Net:
  Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
  #######################################################################
  
  2026-05-04 10:17:34.548907: Using torch.compile...
  2026-05-04 10:17:35.624480: do_dummy_2d_data_aug: False
  2026-05-04 10:17:35.625106: Using splits from existing split file: /workspace/nnUNet_preprocessed/Dataset507_VerSe2020/splits_final.json
  2026-05-04 10:17:35.625324: The split file contains 5 splits.
  2026-05-04 10:17:35.625394: Desired fold for training: 0
  2026-05-04 10:17:35.625459: This split has 208 training and 53 validation cases.
  
  This is the configuration used 

In [46]:
import re
import numpy as np
from pathlib import Path
from datetime import datetime

def parse_log(log_paths):
    """Parsea uno o varios logs de nnUNet."""
    all_epoch_times = []
    all_dice_means  = []
    all_val_losses  = []
    epochs_done     = 0

    for log_path in log_paths:
        with open(log_path) as f:
            content = f.read()
        lines = content.split('\n')

        # Epoch times
        times = re.findall(r'Epoch time: ([\d.]+) s', content)
        all_epoch_times.extend([float(t) for t in times])

        # Epochs completados
        ep = re.findall(r'Epoch (\d+)\n', content)
        if ep:
            epochs_done = max(epochs_done, max(int(e) for e in ep) + 1)

        # Pseudo dice por epoch → promedio de clases válidas
        dice_blocks = re.findall(
            r'Pseudo dice \[([^\]]+)\]', content
        )
        for block in dice_blocks:
            vals = re.findall(r'([\d.]+)\)', block)
            vals = [float(v) for v in vals if float(v) > 0.01]
            if vals:
                all_dice_means.append(np.mean(vals))

        # Val loss
        val_losses = re.findall(r'val_loss ([-\d.]+)', content)
        all_val_losses.extend([float(v) for v in val_losses])

    # Timestamps inicio/fin del primer y último log
    all_logs_sorted = sorted(log_paths)
    with open(all_logs_sorted[0]) as f:
        first_content = f.read()
    with open(all_logs_sorted[-1]) as f:
        last_content = f.read()

    ts_start = re.findall(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', first_content)
    ts_end   = re.findall(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', last_content)

    wall_hours = None
    if ts_start and ts_end:
        fmt = '%Y-%m-%d %H:%M:%S'
        t0  = datetime.strptime(ts_start[0],  fmt)
        t1  = datetime.strptime(ts_end[-1],   fmt)
        wall_hours = round((t1 - t0).total_seconds() / 3600, 2)

    return {
        'Epochs':            epochs_done,
        'Avg epoch (s)':     round(np.mean(all_epoch_times), 1) if all_epoch_times else 0,
        'Total GPU (h)':     round(sum(all_epoch_times) / 3600, 2),
        'Wall time (h)':     wall_hours,
        'Best val Dice':     round(max(all_dice_means), 4) if all_dice_means else None,
        'Final val loss':    round(all_val_losses[-1], 4) if all_val_losses else None,
    }

# ── Parsear los 3 modelos ──────────────────────────────────────────────────
training_stats = {}

for model_name, log_dir in LOG_DIRS.items():
    logs = sorted(Path(log_dir).glob('training_log*.txt'))
    stats = parse_log(logs)
    training_stats[model_name] = stats
    print(f'\n── {model_name} ──────────────────────────────────')
    for k, v in stats.items():
        print(f'  {k:<20}: {v}')

# ── Tabla final ────────────────────────────────────────────────────────────
df_train = pd.DataFrame(training_stats).T
print('\n── Tabla resumen ──────────────────────────────────────')
print(df_train.to_string())
df_train.to_csv(str(OUTPUT_DIR / 'training_stats.csv'))
print(f'\n✓ Guardado → {OUTPUT_DIR}/training_stats.csv')


── nnUNet ──────────────────────────────────
  Epochs              : 0
  Avg epoch (s)       : 51.5
  Total GPU (h)       : 3.57
  Wall time (h)       : 3.66
  Best val Dice       : 0.8511
  Final val loss      : -0.1449

── MedNeXt ──────────────────────────────────
  Epochs              : 0
  Avg epoch (s)       : 109.3
  Total GPU (h)       : 7.59
  Wall time (h)       : 7.65
  Best val Dice       : 1.3825
  Final val loss      : -0.8381

── SwinUNETR ──────────────────────────────────
  Epochs              : 0
  Avg epoch (s)       : 216.7
  Total GPU (h)       : 16.19
  Wall time (h)       : 17.36
  Best val Dice       : 1.0698
  Final val loss      : -0.0886

── Tabla resumen ──────────────────────────────────────
           Epochs  Avg epoch (s)  Total GPU (h)  Wall time (h)  Best val Dice  Final val loss
nnUNet        0.0           51.5           3.57           3.66         0.8511         -0.1449
MedNeXt       0.0          109.3           7.59           7.65         1.3825    